# Global Flood Monitoring (GFM) — flood event analysis

This notebook demonstrates how to search, load, and visualize flood extent data from the [Global Flood Monitoring (GFM)](https://www.globalfloods.eu/) product available on the _eodc_ STAC catalog.

The workflow covers:

1. Searching the STAC catalog for GFM items over a region and time range
2. Exploring flood coverage statistics over time
3. Loading raster data for a specific flood event
4. Visualizing flood extent interactively
5. Exporting and storing data in different ways

More specifically, we want to look at the severe flooding events hitting Mozambique from December 2025 to January 2026.

## 1. Search the STAC catalog

To do so, we first need to import the `pystac` client and specify several parameters for accessing the STAC catalogue: 

- API endpoint
- Collection name
- Name of the asset we are interested in. In our case it is the flood extent, which is a binary layer (0 = no flood, 1 = flood)
- Timespan we are interested in
- Region of interest we are interested in 

In [ ]:
from pystac_client import Client
from datetime import datetime

api_url = "https://stac.eodc.eu/api/v1"
collection_name = "GFM"
asset_name = "ensemble_flood_extent"
timespan = [datetime(2025, 12, 1), datetime(2026, 2, 1)]
roi = [26, -27, 43, -9]

Then we connect to the client and execute the search with the specified spatiotemporal filters.

In [ ]:
eodc_catalog = Client.open(api_url)
search = eodc_catalog.search(
    collections=[collection_name],
    datetime=timespan,
    bbox=roi
)

print("We found", search.matched(), "items, that match our filter criteria.")

Now we can fetch all items with their assets (this takes a bit).

In [ ]:
items = search.item_collection()

Lets look at an item first. Ideally, we want a flood map for each timestamp. Lets browse through the items to identify if timestamps are unique for our region of interest.

In [ ]:
ref_date = items[0].properties["datetime"]
items_date = []
for item in items:
    if  item.properties["datetime"] == ref_date:
        items_date.append(item)

items_date

It seems like they are not! But why? 

The reason is that parts of the GFM processing are tile-based. The Equi7Grid system is used to slice all data globally into smaller portions, so called tiles, which are coregistered, easier to handle and easier to parallelise for processes. More details on the Equi7Grid can be found [here](https://equi7.geo.tuwien.ac.at).

Lets raise our item analysis to the next level and use `pandas` for collecting information and filtering items. In the cell below, we are collecting the observation timestamp as a non timezone-aware `datetime` object, the Equi7Grid tilename, and the coverage ratio of the flood within a tile.

In [ ]:
import pandas as pd

timestamps = []
tilenames = []
flood_coverages = []

tile_edge_length = 300e3 # in metres
sampling = 20 # in metres
n_pixels = (tile_edge_length/sampling)**2 
for item in items:
    timestamp = pd.Timestamp(item.properties["datetime"]).to_pydatetime().replace(tzinfo=None)
    timestamps.append(timestamp)
    tilenames.append(item.properties["Equi7Tile"])
    flood_coverage = item.properties["flooded_pixels"]/n_pixels
    flood_coverages.append(flood_coverage)

stac_df = pd.DataFrame({"timestamp": timestamps, "tilename": tilenames, "item": items, "flood_coverage": flood_coverages})
stac_df

## 2. Explore flood coverage over time

Now we can use the dataframe to filter a single tile of interest to analyse its time series.

In [ ]:
tilename_oi = "AF020M_E066N021T3"
tile_df = stac_df.loc[stac_df["tilename"] == tilename_oi].drop(columns="tilename")
tile_df

Using the new dataframe we can now plot the flood coverage fraction as a bar chart to identify the progression and timing of several flood events within the search period.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 7))
plt.bar(tile_df["timestamp"], tile_df["flood_coverage"])

# tile_df.plot.bar(x="timestamp", y="flood_coverage") would be also an option

## 3. Load raster data for the flood event

After selecting our flood event of interest, we can filter the respective STAC items. 

In [ ]:
event_start = datetime(2026, 1, 15)
event_end = datetime(2026, 1, 18)
event_mask = (tile_df["timestamp"] >= event_start) & (tile_df["timestamp"] <= event_end) 
event_df = tile_df.loc[event_mask]
event_df

The STAC items can now be forwarded to the `stac_load` function of the `odc` package. This function also allows to optionally resample the data. The pixel values of the asset `"ensemble_flood_extent"` are: 

| Value | Meaning |
|-------|---------|
| 0 | Not flooded |
| 1 | Flooded |
| 255 | No data |

In our case we decided to downsample the data to 100m with `mode` resampling to preserve the discrete class values.

In [ ]:
import rioxarray # noqa
from odc.stac import stac_load

data = stac_load(
    event_df["item"], 
    bands=[asset_name],
    resolution=100,
    resampling="mode",
    dtype='uint8',)
data

## 4. Visualize flood extent

The plain data array is not useful for interpreting data, which is why we need some visual context. The following code cell defines an interactive map using [eomaps](https://eomaps.readthedocs.io). We don't need to know the details here - but the map lets us browse through the data and shows only flooded pixels (value `1`) rendered in blue.

In [ ]:
%matplotlib widget
import numpy as np
from eomaps import Maps, widgets
from ipywidgets import VBox
from matplotlib.colors import ListedColormap, BoundaryNorm

# value 0 (non-flooded) → transparent, value 1 (flooded) → nice blue, 255 (nodata) → clipped to over → transparent
FLOOD_CMAP = ListedColormap(["none", "#0077B6"])
FLOOD_CMAP.set_over("none")
FLOOD_NORM = BoundaryNorm([-0.5, 0.5, 1.5], FLOOD_CMAP.N)

def plot_data(dar):
    plt.ioff()  # prevent ipympl from auto-displaying the figure on creation
    m = Maps(figsize=(5, 5), crs=3857)
    m_bg = m.new_layer("all")  # "all" layer is rendered alongside every other layer
    m_bg.add_wms.S2_cloudless.add_layer.s2cloudless_2025_3857(alpha=0.5)
    timestamps = [pd.Timestamp(t).strftime("%Y%m%dT%H%M%S") for t in dar["time"].data]
    for timestamp in timestamps:
        dar_ts = dar.sel(time=timestamp)
        m_data = m.new_layer(layer=timestamp)
        m_data.set_data(
            data=dar_ts.values.T,
            x=dar_ts.x.values,
            y=dar_ts.y.values,
            crs=3857,
        )
        m_data.set_shape.shade_raster()
        m_data.plot_map(cmap=FLOOD_CMAP, norm=FLOOD_NORM, set_extent=True)
        n_flooded_pxs = int(np.sum(dar_ts.data == 1))
        m_data.add_title(f"Flood map on {timestamp} (flooded pixels: {n_flooded_pxs})")

    layer_selector = widgets.LayerSelectionSlider(m, layers=timestamps)
    plt.ion()
    display(VBox([layer_selector, m.f.canvas]))
    m.show_layer(timestamps[0])

Before starting the visualisation, we reproject the data to Web Mercator ("EPSG:3857") to match the projection of the background map. This is often faster than re-projecting and re-rendering the data on the fly.

In [ ]:
data_plot = data.rio.reproject("EPSG:3857")
plot_data(data_plot["ensemble_flood_extent"])

## 5. Export raster data of a flood event

Maybe we found a specific flood observation we want to inspect in more detail or analyse it together with other data sources. There are several options to download/retrieve the cloud-optimized GeoTIFF data.

As a first option, we can directly download the data using Python's `urllib` package.

In [ ]:
import urllib.request

url_path = event_df.loc[679]["item"].assets[asset_name].href
urllib.request.urlretrieve(url_path, "my_flood_of_interest.tif")

An alternative is to load the data via `rioxarray` and `rasterio` into memory in case we want to analyse or modify it in Python first.

In [ ]:
ds = rioxarray.open_rasterio(url_path)
ds

This gives us the usual powerful `xarray` interface to the data. Finally, this also allows us to write the data with `to_raster` to disk.

In [ ]:
ds.rio.to_raster("my_flood_of_interest_rio.tif")